In [2]:
import os
import sys
from pathlib import Path
import pandas as pd

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


import nvitk as nv

from nvitk import db
from nvitk.db import DataRepo, import_pesabrain_source
from nvitk.db import upsert_nifti_assets, upsert_dicom_assets, register_nifti_tree, register_dicom_tree
from nvitk.db.repo import DataRepo
from nvitk.db.xnat import XnatConnectionConfig, sync_xnat_project
from nvitk.db import get_repo_from_settings

In [3]:
# !python test/prototypes/db/import_new_vars.py --dataset-root ~/nvitk/dataset/nvitk-dataset/ --build-sqlite-index

In [4]:
repo, xnat_config = get_repo_from_settings(return_xnat_config=True)

Using local root: ~/nvitk/dataset/nvitk-dataset


In [5]:
repo.clinical(
    filters={"variable_id": ["bpxdim", "pp", 'bpxsym', 'pulse_pressure_map', 'apoe_group', 'apoe']},
)

/home/imarcoss/nvitk/src/nvitk/db/repo.py:599: UserWarning: Error querying table clinical_measurements with SQLite index. Falling back to parquet read.
  df = self._load_table_frame(
/home/imarcoss/nvitk/src/nvitk/db/repo.py:427: UserWarning: Error querying table cohort_membership with SQLite index. Falling back to parquet read.
  cm = self._load_table_frame(


,subject_uid,visit_id,apoe,apoe_group,bpxdim,bpxsym,pp,pulse_pressure_map
0,PESA10017225,4,E3/E3,only_e3,82.0,130.0,48.0,98.0
1,PESA100489,4,E3/E4,any_e4,77.0,128.0,51.0,94.0
2,PESA10048900,4,E3/E3,only_e3,79.0,150.0,71.0,102.666667
3,PESA10055241,4,E3/E4,any_e4,70.0,117.0,47.0,85.666667
4,PESA1006009,4,E3/E3,only_e3,75.0,124.0,49.0,91.333333
...,...,...,...,...,...,...,...,...
996,PESA9928801,4,E3/E3,only_e3,73.0,104.0,31.0,83.333333
997,PESA9935104,4,E3/E3,only_e3,76.0,122.0,46.0,91.333333
998,PESA9947716,4,E3/E3,only_e3,71.0,114.0,43.0,85.333333
999,PESA9954025,4,E3/E3,only_e3,79.0,126.0,47.0,94.666667


In [6]:
repo.image(
    modality="flair",
    variables=['wmh_dist', 'wmh_freq', 'wmh_les', 'wmh_reg']
)

/home/imarcoss/nvitk/src/nvitk/db/repo.py:427: UserWarning: Error querying table cohort_membership with SQLite index. Falling back to parquet read.
  cm = self._load_table_frame(


,subject_uid,session_id,1_wmh_dist,1_wmh_freq,1_wmh_les,1_wmh_reg,2_wmh_dist,2_wmh_freq,2_wmh_les,2_wmh_reg,...,t_wmh_les,t_wmh_reg,tl_wmh_dist,tl_wmh_freq,tl_wmh_les,tl_wmh_reg,tr_wmh_dist,tr_wmh_freq,tr_wmh_les,tr_wmh_reg
0,PESA10017225,BMRI613225,0.320745,0.106074,2839.708513,26771.0,0.22162,0.032775,1962.101758,59866.0,...,1393.926699,115106.0,0.059321,0.009252,525.199466,56765.0,0.098123,0.014891,868.727233,58341.0
1,PESA100489,BMRI649274,0.295037,0.14012,3586.933487,25599.0,0.170668,0.035126,2074.909489,59071.0,...,1768.910992,123011.0,0.088005,0.017171,1069.929088,62312.0,0.057494,0.011516,698.981903,60699.0
2,PESA10055241,BMRI872431,0.580707,0.109367,2990.434142,27343.0,0.195301,0.017058,1005.731797,58958.0,...,889.905221,112233.0,0.080859,0.007641,416.393687,54493.0,0.09195,0.008201,473.511535,57740.0
3,PESA1006009,BMRI923674,0.399685,0.085393,1611.367201,18870.0,0.160635,0.012939,647.615519,50052.0,...,373.542055,104225.0,0.036069,0.002877,145.417255,50536.0,0.056584,0.004249,228.1248,53689.0
4,PESA10061584,BMRI287356,0.319367,0.02685,727.475864,27094.0,0.280982,0.010577,640.038536,60512.0,...,107.059194,117811.0,0.037943,0.001436,86.428205,60204.0,0.009057,0.000358,20.630989,57607.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968,PESA9928801,BMRI487416,0.533885,0.052481,971.736903,18516.0,0.124276,0.004945,226.197829,45740.0,...,355.542384,92910.0,0.082011,0.003234,149.269458,46163.0,0.113329,0.004413,206.272926,46747.0
969,PESA9935104,BMRI881199,0.393805,0.032496,689.144976,21207.0,0.240798,0.00885,421.38811,47614.0,...,292.116036,100050.0,0.078873,0.002772,138.024824,49786.0,0.088054,0.003066,154.091211,50264.0
970,PESA9947716,BMRI447102,0.535515,0.080649,2119.787922,26284.0,0.126043,0.00869,498.928012,57413.0,...,589.57518,111511.0,0.08794,0.00644,348.103547,54055.0,0.061002,0.004203,241.471634,57456.0
971,PESA9954025,BMRI971103,0.407106,0.048519,1395.791544,28768.0,0.22159,0.011748,759.736372,64672.0,...,1152.374893,124869.0,0.146212,0.008055,501.297463,62235.0,0.189898,0.010395,651.07743,62634.0


In [7]:
# c = repo.image(
#     modality="flair",
#     variables=['wmh_dist', 'wmh_freq', 'wmh_les', 'wmh_reg']
# ).columns
# [print(col) for col in c]

In [8]:
repo.image(
    modality="t1",
    filters={}
)

/home/imarcoss/nvitk/src/nvitk/db/repo.py:427: UserWarning: Error querying table cohort_membership with SQLite index. Falling back to parquet read.
  cm = self._load_table_frame(


,subject_uid,session_id,id_t1_cortical_volume,id_t1_subcortical_volume,measurement_t1_cortical_volume,measurement_t1_subcortical_volume,region_t1_cortical_volume,region_t1_subcortical_volume,side_t1_cortical_volume,subject_t1_cortical_volume,subject_t1_subcortical_volume,value_t1_cortical_volume,value_t1_subcortical_volume
0,PESA10017225,BMRI613225,CNIC_E01035,CNIC_E01035,MeanCurv,NVoxels,superiorfrontal,Right-VentralDC,right,PESA10017225,PESA10017225,0.113,4187.0
1,PESA1002001,IA706156,CNIC_E00484,CNIC_E00484,MeanCurv,NVoxels,superiorparietal,Right-vessel,right,PESA1002001,PESA1002001,0.108,41.0
2,PESA100489,BMRI649274,CNIC_E00957,CNIC_E00957,MeanCurv,NVoxels,superiorparietal,Right-vessel,right,PESA100489,PESA100489,0.112,20.0
3,PESA10055241,BMRI872431,CNIC_E00903,CNIC_E00903,CurvInd,normRange,temporalpole,CC_Central,left,PESA10055241,PESA10055241,0.8,77.0
4,PESA1006009,BMRI923674,CNIC_E00301,CNIC_E00301,MeanCurv,NVoxels,superiorparietal,Right-vessel,right,PESA1006009,PESA1006009,0.11,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,PESA9935104,BMRI881199,CNIC_E00357,CNIC_E00357,MeanCurv,NVoxels,superiorfrontal,Right-VentralDC,right,PESA9935104,PESA9935104,0.115,3744.0
1378,PESA9947716,BMRI447102,CNIC_E00399,CNIC_E00399,MeanCurv,NVoxels,superiortemporal,Right-choroid-plexus,right,PESA9947716,PESA9947716,0.101,606.0
1379,PESA9954025,BMRI971103,CNIC_E01530,CNIC_E01530,MeanCurv,NVoxels,superiorparietal,Right-WM-hypointensities,right,PESA9954025,PESA9954025,0.115,0.0
1380,PESA996004,IA192427,CNIC_E01220,CNIC_E01220,MeanCurv,NVoxels,superiorfrontal,Right-WM-hypointensities,right,PESA996004,PESA996004,0.124,0.0


In [ ]:
repo.image(
    modality="asl",
    variables=['att_mean']
)

/home/imarcoss/nvitk/src/nvitk/db/repo.py:427: UserWarning: Error querying table cohort_membership with SQLite index. Falling back to parquet read.
  cm = self._load_table_frame(


,subject_uid,session_id,3rd_ventricle_att_cov,3rd_ventricle_att_mean,4th_ventricle_att_cov,4th_ventricle_att_mean,brain_stem_att_cov,brain_stem_att_mean,cc_anterior_att_cov,cc_anterior_att_mean,...,right_putamen_att_cov,right_putamen_att_mean,right_thalamus_att_cov,right_thalamus_att_mean,right_ventraldc_att_cov,right_ventraldc_att_mean,right_vessel_att_cov,right_vessel_att_mean,wm_hypointensities_att_cov,wm_hypointensities_att_mean
0,PESA10017225,BMRI613225,0.257669,1.437767,0.253522,1.225247,0.259966,1.236855,0.294619,1.135062,...,0.294442,0.969021,0.218772,1.221106,0.291259,1.204374,NaN,NaN,0.238617,1.333060
1,PESA1002001,IA706156,0.283619,1.323019,0.193815,1.264247,0.238499,1.243192,0.229492,1.190722,...,0.313567,1.091082,0.269387,1.276714,0.283909,1.209576,NaN,NaN,0.193788,1.328825
2,PESA100489,BMRI649274,0.219315,1.324065,0.274814,1.261800,0.246795,1.267750,0.398508,1.238174,...,0.270976,1.042300,0.211487,1.361830,0.247684,1.261054,NaN,NaN,0.278167,1.342592
3,PESA10055241,BMRI872431,0.202288,1.340492,0.269942,1.241811,0.266264,1.316504,0.283528,1.210026,...,0.332172,1.146493,0.245452,1.306093,0.277143,1.383056,0.029544,1.353628,0.221386,1.281596
4,PESA1006009,BMRI923674,0.201451,1.380013,0.230689,1.301129,0.258987,1.253961,0.356937,1.023581,...,0.322995,1.073846,0.198996,1.334591,0.236346,1.226049,NaN,NaN,0.260136,1.317092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1335,PESA9935104,BMRI881199,0.209957,1.249058,0.203662,1.236699,0.274440,1.137976,0.221665,1.271728,...,0.286074,1.080458,0.233065,1.106522,0.311487,1.133721,0.000000,0.990150,0.239889,1.317866
1336,PESA9947716,BMRI447102,0.214146,1.370706,0.196591,1.259955,0.231038,1.189831,0.270074,1.165071,...,0.287167,1.004282,0.199218,1.235063,0.249222,1.175415,0.256725,0.661932,0.229334,1.369928
1337,PESA9954025,BMRI971103,0.214638,1.285378,0.226017,1.354978,0.270284,1.186844,0.353798,1.003245,...,0.305693,0.988986,0.211730,1.231453,0.277484,1.184928,0.000000,1.252980,0.189115,1.246740
1338,PESA996004,IA192427,0.280942,1.277582,0.204224,1.259212,0.221675,1.353739,0.355473,1.154819,...,0.299083,1.207571,0.242257,1.331145,0.231756,1.289496,0.133067,0.877926,0.217072,1.380833


In [10]:
# repo.build_sqlite_index()
# repo.drop_table("cognitive_measurements")

In [11]:
repo.cognitive(
    # variables=['z_compo_processingspeed']
)

/home/imarcoss/nvitk/src/nvitk/db/repo.py:427: UserWarning: Error querying table cohort_membership with SQLite index. Falling back to parquet read.
  cm = self._load_table_frame(


,subject_uid,visit_id,age_at_cog_months,age_at_cog_years,cdr_activdomesticas,cdr_activfueradecasa,cdr_cuidadopersonal,cdr_global,cdr_memoria,cdr_orientacion,...,wais_puzles,wais_span_creciente,wais_span_directo,wais_span_inverso,z_compo_attworkmemospeed,z_compo_epismemory,z_compo_globalfull,z_compo_pacc_bbrc,z_compo_pacc_bbrc_plus,z_compo_processingspeed
0,PESA10017225,4,618.766667,51.562842,0.0,0.0,0.0,0.0,0.0,0.0,...,17.0,8.0,6.0,4.0,0.171771,1.128244,0.768365,1.005172,0.729974,-0.074405
1,PESA1002001,4,655.064516,54.586301,0.0,0.0,0.0,0.0,0.0,0.0,...,12.0,8.0,7.0,7.0,3.242329,0.942087,2.176751,2.534507,2.341827,2.363359
2,PESA100489,4,691.5,57.622951,0.0,0.0,0.0,0.0,0.0,0.0,...,17.0,6.0,7.0,5.0,0.340117,1.55591,1.234198,1.15436,1.21868,-0.298993
3,PESA10048900,4,775.290323,64.605479,0.0,0.0,0.0,0.0,0.0,0.0,...,18.0,4.0,5.0,4.0,-2.266308,-0.932193,-1.731374,-1.881857,-1.655297,-1.69753
4,PESA10055241,4,773.0,64.413699,0.5,0.0,1.0,0.5,0.5,0.0,...,6.0,6.0,5.0,4.0,-1.065348,-2.638478,-2.522402,-2.40041,-2.540746,-0.962352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1373,PESA9935104,4,729.548387,60.794521,0.0,0.0,0.0,0.0,0.0,0.0,...,11.0,7.0,5.0,4.0,-0.294657,-1.795589,-1.474183,-1.187782,-1.365462,0.390962
1374,PESA9947716,4,701.37931,58.446575,0.0,0.0,0.0,0.0,0.0,0.0,...,16.0,6.0,6.0,7.0,1.437316,0.566501,1.056801,0.121655,0.385807,1.191238
1375,PESA9954025,4,624.448276,52.035519,0.0,0.0,0.0,0.0,0.0,0.0,...,21.0,6.0,7.0,4.0,-0.400128,0.466686,0.391403,0.038178,0.47512,0.178901
1376,PESA996004,4,688.387097,57.367123,0.0,0.0,0.0,0.0,0.0,0.0,...,15.0,6.0,7.0,6.0,0.397587,0.109454,0.375277,-0.017019,0.341128,-0.00232
